In [84]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [85]:
DATA = "../data/raw/listings.csv"
OUTPUT = "../data/processed/cleanedr1.csv"
REPORT = "../reports/"

In [86]:
df = pd.read_csv(DATA)
init_count = len(df)

In [101]:
# Price vs Bedrooms
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(df["bedrooms"], df["price"], alpha=0.3, s=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel("Bedrooms")
axes[0].set_ylabel("Price")
axes[0].set_title("Price vs Bedrooms (Linear Scale)")

v_prices = df[df["price"] > 0]
axes[1].scatter(v_prices["bedrooms"], v_prices["price"], alpha=0.3, s=10)
axes[1].set_xlabel("Bedrooms")
axes[1].set_ylabel("Price (log scale)")
axes[1].set_yscale("log")
axes[1].set_title("Price vs Bedrooms (Log Scale)")
axes[1].grid(True, alpha=0.3)

price_quantiles = df["price"].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
for q in price_quantiles:
    axes[0].axhline(q, color="red", alpha=0.2, linestyle="--")
    axes[1].axhline(q, color="red", alpha=0.2, linestyle="--")

plt.tight_layout()
plt.savefig(f"{REPORT}01_price_bedrooms.png", dpi=150)
plt.close()
print("✓ Saved price vs bedrooms inspection")


✓ Saved price vs bedrooms inspection


In [102]:
plt.figure(figsize=(12, 6))
df.boxplot(column="price", by="house_type", ax=plt.gca())
plt.yscale("log")
plt.xticks(rotation=45, ha="right")
plt.title("Price Distribution by Property Type (Log Scale)")
plt.suptitle("")
plt.ylabel("Price (log)")
plt.tight_layout()
plt.savefig(f"{REPORT}01_price_vs_property_type.png", dpi=150)
plt.close()
print("✓ Saved price vs property type inspection")

✓ Saved price vs property type inspection


In [103]:
top_neighborhoods = df["location"].value_counts().head(30).index
df_top = df[df["location"].isin(top_neighborhoods)]

plt.figure(figsize=(14, 8))
df_top.boxplot(column="price", by="location", ax=plt.gca())
plt.yscale("log")
plt.xticks(rotation=45, ha="right")
plt.title("Price Distribution by Neighborhood (Top 20, Log Scale)")
plt.suptitle("")
plt.ylabel("Price (log)")
plt.tight_layout()
plt.savefig(f"{REPORT}01_price_vs_neighborhood.png", dpi=150)
plt.close()
print("✓ Saved price vs neighborhood inspection")

✓ Saved price vs neighborhood inspection


In [104]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Overall histogram
axes[0, 0].hist(df['price'], bins=100, edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("Price")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Price Distribution - Looking for Spikes")
axes[0, 0].axvline(df['price'].median(), color="red", label="Median", linestyle="--")
axes[0, 0].legend()

# Log histogram
axes[0, 1].hist(
    np.log10(df[df['price'] > 0]['price']), bins=100, edgecolor="black", alpha=0.7
)
axes[0, 1].set_xlabel("Log10(Price)")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].set_title("Log Price Distribution - Looking for Modes")

# Suspicious price values
price_counts = df['price'].value_counts().head(20)
axes[1, 0].barh(range(len(price_counts)), price_counts.values)
axes[1, 0].set_yticks(range(len(price_counts)))
axes[1, 0].set_yticklabels([f"${p:,.0f}" for p in price_counts.index])
axes[1, 0].set_xlabel("Count")
axes[1, 0].set_title("Top 20 Most Common Exact Prices (Suspicious?)")

# Price percentiles
percentiles = range(0, 101, 5)
price_percentiles = [df['price'].quantile(p / 100) for p in percentiles]
axes[1, 1].plot(percentiles, price_percentiles, marker="o")
axes[1, 1].set_xlabel("Percentile")
axes[1, 1].set_ylabel("Price")
axes[1, 1].set_title("Price Percentile Curve - Looking for Jumps")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{REPORT}01_price_patterns.png", dpi=150)
plt.close()
print("✓ Saved detailed price pattern inspection")


✓ Saved detailed price pattern inspection


In [105]:
print("\n--- PRICE STATISTICS FOR THRESHOLD DECISIONS ---")
print(f"Minimum price: ${df['price'].min():,.2f}")
print(f"1st percentile: ${df['price'].quantile(0.01):,.2f}")
print(f"5th percentile: ${df['price'].quantile(0.05):,.2f}")
print(f"25th percentile: ${df['price'].quantile(0.25):,.2f}")
print(f"Median: ${df['price'].quantile(0.50):,.2f}")
print(f"75th percentile: ${df['price'].quantile(0.75):,.2f}")
print(f"95th percentile: ${df['price'].quantile(0.95):,.2f}")
print(f"99th percentile: ${df['price'].quantile(0.99):,.2f}")
print(f"Maximum price: ${df['price'].max():,.2f}")



--- PRICE STATISTICS FOR THRESHOLD DECISIONS ---
Minimum price: $150.00
1st percentile: $500.00
5th percentile: $900.00
25th percentile: $2,500.00
Median: $5,000.00
75th percentile: $15,000.00
95th percentile: $48,000.00
99th percentile: $91,000.00
Maximum price: $1,200,000,000.00


In [106]:
suspicious_values = [1, 10, 100, 1000, 9999, 99999]
for val in suspicious_values:
    count = (df['price'] == val).sum()
    if count > 0:
        print(f"  ⚠ Exact price = ${val}: {count} listings")


  ⚠ Exact price = $1000: 276 listings


In [93]:
df = df[df['price']!=9999]

In [94]:
df = df[df['bedrooms']<=10]

In [95]:
df = df[~((df['price']>=50000) & (df['bedrooms']==1))]

In [96]:
df = df[~((df["price"] >= 100000) & (df["bedrooms"] == 2))]


In [97]:
df = df[~((df["price"] >= 100000) & (df["bedrooms"] == 3))]


In [107]:
df[df['price']>=100000]

,Unnamed: 0,url,fetch_date,title,location,house_type,bathrooms,bedrooms,price,Property Address,...,Kitchen Cabinets,Kitchen Shelf,Microwave,Pop Ceiling,Pre-Paid Meter,Refrigerator,TV,Tiled Floor,Wardrobe,Wi-Fi
546,546,https://jiji.com.gh/dome/houses-apartments-for...,2025-12-25 20:37:55.132964,4bdrm House in Dome for rent,Dome,House,4,4,5250000.0,dome,...,0,0,0,0,0,0,0,0,0,0
654,654,https://jiji.com.gh/airport-residential-area/h...,2025-12-25 20:37:52.833941,"4bdrm House in Chain Homes, Airport for rent",Airport Residential Area,House,5,4,128000.0,Greater Accra,...,0,0,0,0,0,0,0,0,0,0
668,668,https://jiji.com.gh/airport-residential-area/h...,2025-12-25 20:37:52.805595,6bdrm House in Airport Residential for rent,Airport Residential Area,House,7,6,130000.0,Greater Accra,...,0,0,0,0,0,0,0,0,0,0
812,812,https://jiji.com.gh/acquinas/houses-apartments...,2025-12-25 20:37:50.358536,"4bdrm House in Cantonments, Acquinas for rent",Cantonments,House,5,4,156000.0,GREATER ACCRA,...,0,0,0,0,0,0,0,0,0,0
824,824,https://jiji.com.gh/celsbridge-area/houses-apa...,2025-12-25 20:37:50.236680,"4bdrm House in Labone, Celsbridge Area for rent",Labone,House,5,4,120000.0,Greater Accra,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13639,13640,https://jiji.com.gh/cantonments/houses-apartme...,2025-12-25 20:38:34.197568,Furnished 4bdrm House in Cantonments for rent,Cantonments,House,4,4,100000.0,American Embassy,...,0,0,0,0,0,0,1,0,0,0
13682,13683,https://jiji.com.gh/community-25/houses-apartm...,2025-12-25 20:38:33.521861,Furnished 4bdrm Mansion in Community 6 for rent,Tema Metropolitan,Mansion,4,4,2869367.0,"Box 346, Tema",...,0,0,0,0,0,0,0,0,0,0
13739,13740,https://jiji.com.gh/ablekuma/houses-apartments...,2025-12-25 20:38:31.407781,"4bdrm Apartment in The Survivor Villa, Ablekum...",Ablekuma,Apartment,4,4,1641804.0,Ablekuma Phobia,...,0,0,0,0,0,0,0,0,0,0
14104,14105,https://jiji.com.gh/kasoa/houses-apartments-fo...,2025-12-25 20:38:26.766741,"4bdrm House in Rab Estates, Kasoa for rent",Kasoa,House,4,4,900000.0,Kasoa Ashalaja,...,0,0,0,0,0,0,0,0,0,0
